In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score

# ----------------------------
# 🔍 Root directory and file search
# ----------------------------
root_dir = Path("/research/projects/Sahika/projects/liver_PDFF/clean_code/runs/swin_tiny_patch4_window7_224.ms_in1k_relu/700frame_window2100_skip3/")
csv_files = list(root_dir.rglob("*_predicted_true_values.csv"))

# ----------------------------
# 📊 Collect metrics from all files
# ----------------------------
records = []

for csv_path in csv_files:
    model_name = csv_path.parent.name  # define early so it's available in except
    set_name = "unknown"
    try:
        df = pd.read_csv(csv_path)

        if csv_path.name.startswith("val"):
            set_name = "val"
        elif csv_path.name.startswith("test"):
            set_name = "test"

        mae = np.abs(df["target"] - df["prediction"]).mean()

        y_true = (df["target"] > 5).astype(int)
        y_pred = (df["prediction"] > 5).astype(int)

        f1 = f1_score(y_true, y_pred)

        tp = ((y_true == 1) & (y_pred == 1)).sum()
        fn = ((y_true == 1) & (y_pred == 0)).sum()
        fp = ((y_true == 0) & (y_pred == 1)).sum()

        recall    = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan

        records.append({
            "model": model_name,
            "set": set_name,
            "mae": round(mae, 3),
            "f1": round(f1, 3),
            "FN": int(fn),
            "FP": int(fp),
            "recall": round(recall, 3) if pd.notna(recall) else np.nan,
            "precision": round(precision, 3) if pd.notna(precision) else np.nan
        })

    except Exception as e:
        print(f"[ERROR] Failed to process {csv_path}: {e}")
        records.append({
            "model": model_name,
            "set": set_name,
            "mae": np.nan,
            "f1": np.nan,
            "FN": np.nan,
            "FP": np.nan,
            "recall": np.nan,
            "precision": np.nan
        })

# ----------------------------
# 🧮 Final Results and Sorting (VAL recall → TEST follow same order)
# ----------------------------
results_df = pd.DataFrame(records)

val_df  = results_df[results_df["set"] == "val"].copy()
test_df = results_df[results_df["set"] == "test"].copy()

if not val_df.empty:
    # 1) Sort VAL by recall (desc)
    val_df = val_df.sort_values(["f1", "model"], ascending=[False, True], na_position="last").reset_index(drop=True)

    # 2) Build model order from VAL
    model_order = val_df["model"].tolist()

    # 3) Reorder TEST to follow the same model order
    test_df["model_cat"] = pd.Categorical(test_df["model"], categories=model_order, ordered=True)
    aligned_test = test_df[~test_df["model_cat"].isna()].sort_values("model_cat").drop(columns="model_cat")
    # models only in TEST → append at the end alphabetically
    extra_test = test_df[test_df["model_cat"].isna()].drop(columns="model_cat").sort_values("model")

    test_df_ordered = pd.concat([aligned_test, extra_test], ignore_index=True)

    # 4) Combine: VAL on top, TEST below (same order)
    final_df = pd.concat([val_df, test_df_ordered], axis=0).reset_index(drop=True)
else:
    # Fallback: no VAL rows → just sort TEST by recall desc
    test_df = test_df.sort_values(["f1", "model"], ascending=[False, True], na_position="last").reset_index(drop=True)
    final_df = test_df.copy()

# 💾 Kaydet
final_df.to_csv(root_dir / "all_results.csv", index=False)

# 📤 Göster
final_df


,model,set,mae,f1,FN,FP,recall,precision
0,mh_gated8,val,2.820,0.889,1,1,0.889,0.889
1,transf8,val,3.736,0.889,1,1,0.889,0.889
2,mean,val,3.310,0.875,2,0,0.778,1.000
3,tconv32_gated8,val,3.027,0.875,2,0,0.778,1.000
4,tconv16_gated4,val,3.637,0.857,0,3,1.000,0.750
5,attention,val,3.021,0.824,2,1,0.778,0.875
6,gated,val,3.270,0.824,2,1,0.778,0.875
7,mh_gated4,val,2.652,0.824,2,1,0.778,0.875
8,tconv32_gated4,val,3.679,0.824,2,1,0.778,0.875
9,mh_gated16,val,3.705,0.818,0,4,1.000,0.692


In [11]:
final_df[final_df['set'] == 'test']

,model,set,mae,f1,FN,FP,recall,precision
10,mh_gated16,test,3.560,1.000,0,0,1.000,1.000
11,tconv64_gated8,test,5.339,0.686,1,10,0.923,0.545
12,attention,test,3.760,0.800,3,2,0.769,0.833
13,max,test,4.021,0.880,2,1,0.846,0.917
14,mh_gated4,test,3.348,0.815,2,3,0.846,0.786
15,tconv32_gated8,test,3.458,0.929,0,2,1.000,0.867
16,gated,test,4.054,0.818,4,0,0.692,1.000
17,gru,test,5.921,0.522,7,4,0.462,0.600
18,mean,test,4.222,0.762,5,0,0.615,1.000
19,tconv16_gated4,test,4.653,0.556,8,0,0.385,1.000
